# Jamboree Admission Prediction – Linear Regression Pipeline

This notebook demonstrates the complete ML pipeline with clear, interview-ready comments.

In [2]:

# ===============================
# 1. Import Required Libraries
# ===============================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats


In [3]:

# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("Jamboree_Admission.csv")

df.head()


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,1,337,118,4,4.5,4.5,9.65,1,0.92
1,2,324,107,4,4.0,4.5,8.87,1,0.76
2,3,316,104,3,3.0,3.5,8.00,1,0.72
3,4,322,110,3,3.5,2.5,8.67,1,0.80
4,5,314,103,2,2.0,3.0,8.21,0,0.65


In [4]:

# ===============================
# 3. Data Quality Checks
# ===============================
df.shape
df.describe().T
df.info()
df.isnull().sum()
df[df.duplicated()].count()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Serial No.         500 non-null    int64  
 1   GRE Score          500 non-null    int64  
 2   TOEFL Score        500 non-null    int64  
 3   University Rating  500 non-null    int64  
 4   SOP                500 non-null    float64
 5   LOR                500 non-null    float64
 6   CGPA               500 non-null    float64
 7   Research           500 non-null    int64  
 8   Chance of Admit    500 non-null    float64
dtypes: float64(4), int64(5)
memory usage: 35.3 KB


Serial No.           0
GRE Score            0
TOEFL Score          0
University Rating    0
SOP                  0
LOR                  0
CGPA                 0
Research             0
Chance of Admit      0
dtype: int64

In [8]:
df

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337,118,4,4.5,4.5,9.65,1,0.92
1,324,107,4,4.0,4.5,8.87,1,0.76
2,316,104,3,3.0,3.5,8.00,1,0.72
3,322,110,3,3.5,2.5,8.67,1,0.80
4,314,103,2,2.0,3.0,8.21,0,0.65
...,...,...,...,...,...,...,...,...
495,332,108,5,4.5,4.0,9.02,1,0.87
496,337,117,5,5.0,5.0,9.87,1,0.96
497,330,120,5,4.5,5.0,9.56,1,0.93
498,312,103,4,4.0,5.0,8.43,0,0.73


In [ ]:

# ===============================
# 4. Drop Identifier Column
# ===============================
df.drop(columns=['Serial No.'], inplace=True)


In [9]:

# ===============================
# 5. Clean Column Names
# ===============================
df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
df.columns


Index(['gre_score', 'toefl_score', 'university_rating', 'sop', 'lor', 'cgpa',
       'research', 'chance_of_admit'],
      dtype='object')

In [10]:

# ===============================
# 6. Feature Categorization
# ===============================
cat_cols = ['university_rating', 'sop', 'lor', 'research']
num_cols = ['gre_score', 'toefl_score', 'cgpa']
target = 'chance_of_admit'


In [ ]:

# ===============================
# 7. Baseline Model (All Features)
# ===============================
X = df.drop(columns=[target])
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


In [11]:

# ===============================
# 8. VIF Calculation
# ===============================
def calculate_vif(dataframe):
    vif_df = pd.DataFrame()
    vif_df["Feature"] = dataframe.columns
    vif_df["VIF"] = [
        variance_inflation_factor(dataframe.values, i)
        for i in range(dataframe.shape[1])
    ]
    return vif_df

calculate_vif(df.drop(columns=[target]))


,Feature,VIF
0,gre_score,1308.061089
1,toefl_score,1215.951898
2,university_rating,20.933361
3,sop,35.265006
4,lor,30.911476
5,cgpa,950.817985
6,research,2.869493


In [ ]:

# ===============================
# 9. Feature Selection (After VIF)
# ===============================
X = df[['cgpa', 'research']]
y = df[target]

X_scaled = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


In [ ]:

# ===============================
# 10. Final Linear Regression Model
# ===============================
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print("Test R2 Score:", round(r2, 2))


In [ ]:

# ===============================
# 11. Residual Diagnostics
# ===============================
residuals = y_test - y_pred

sns.histplot(residuals, kde=True)
plt.show()

stats.probplot(residuals, plot=plt)
plt.show()

plt.scatter(y_pred, residuals)
plt.axhline(0, color='red')
plt.show()
